# Pose estimation using Sapiens2 (model 0.4b)

For motion trackong, we use Sapiens 2 (ref)

- Sapiens2 repo: https://github.com/facebookresearch/sapiens2
- Sapiens2 models: https://huggingface.co/facebook/sapiens2

## Prepare environment

In [1]:
import os, glob, cv2, json, shutil, subprocess

curfolder = os.getcwd()

videodata = glob.glob(os.path.join(curfolder, "GTST_sampled", "*.mp4"))
print(len(videodata))

1499


## Configure model (Sapiens2)

In [18]:
PYTHON        = r"C:\Users\kadava\AppData\Local\anaconda3\envs\sapiens2\python.exe"  # sapiens2 env
SAPIENS2_ROOT = curfolder
POSE_DIR      = os.path.join(SAPIENS2_ROOT, "sapiens", "pose")   # run vis_pose.py from here
CKPT_ROOT     = os.path.join(curfolder, "sapiens2_host")

MODEL_NAME = "sapiens2_0.4b"          
DETECTOR   = os.path.join(CKPT_ROOT, "detector", "detr-resnet-101-dc5")
CHECKPOINT = os.path.join(CKPT_ROOT, "pose", f"{MODEL_NAME}_pose.safetensors")
CONFIG     = f"configs/keypoints308/shutterstock_goliath_3po/{MODEL_NAME}_keypoints308_shutterstock_goliath_3po-1024x768.py"  # relative to POSE_DIR

OUTROOT    = os.path.join(SAPIENS2_ROOT, "Sapiens_Output")   # one subfolder per video

vfiles = videodata

def runcommand(command, cwd=None):
    r = subprocess.run(command, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0:
        print("STDERR:\n", r.stderr[-3000:])   # the real traceback
        print("STDOUT:\n", r.stdout[-1000:])
    return r

## Loop over videos

In [ ]:
import pandas as pd

missing = pd.read_csv("missing_kinematics.csv")

In [11]:
import shutil

out_folder = os.path.join(curfolder, "missing_kinematics_videos")
os.makedirs(out_folder, exist_ok=True)

copied, not_found = [], []
for row in missing.itertuples():
    src = row.path
    # replace f: by g: in path
    src = src.replace("f:", "g:")
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(out_folder, row.file))   # copy2 keeps timestamps
        copied.append(row.file)
    else:
        not_found.append(src)

print(f"copied {len(copied)} of {len(missing)} videos to {out_folder}")
if not_found:
    print(f"{len(not_found)} source videos not found:")
    print("\n".join(not_found))

copied 78 of 78 videos to g:\GtST_FLESH_KinTrajectory\missing_kinematics_videos


In [20]:
vfiles = glob.glob(os.path.join(out_folder, "*.mp4"))

In [21]:
len(vfiles)

78

In [17]:
print(repr(POSE_DIR), os.path.isdir(POSE_DIR))

'g:\\GtST_FLESH_KinTrajectory\\sapiens\\pose' False


In [22]:
import subprocess, threading, time

sapiens_error = []
FRAME_TIMEOUT = 600  # 10 min per frame absolute ceiling; total timeout set per-video below

def run_sapiens(command, cwd, total_frames, per_frame_timeout=600, poll_interval=5):
    """
    Run Sapiens subprocess with a dynamic timeout based on frame count.
    Watches posedir for new .json files; if none appear for per_frame_timeout
    seconds, kills the process and returns a timeout result.
    Returns (returncode, timed_out: bool).
    """
    total_timeout = total_frames * per_frame_timeout  # generous ceiling
    proc = subprocess.Popen(command, shell=True, cwd=cwd,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    stdout_lines, stderr_lines = [], []

    def drain(pipe, store):
        for line in pipe:
            store.append(line)
    threading.Thread(target=drain, args=(proc.stdout, stdout_lines), daemon=True).start()
    threading.Thread(target=drain, args=(proc.stderr, stderr_lines), daemon=True).start()

    deadline = time.time() + total_timeout
    last_json_count = 0
    stall_since = time.time()

    while proc.poll() is None:
        time.sleep(poll_interval)
        # count json files written so far (progress signal)
        current_jsons = len(glob.glob(os.path.join(posedir, "*.json")))
        if current_jsons > last_json_count:
            last_json_count = current_jsons
            stall_since = time.time()  # reset stall clock on progress

        stalled = (time.time() - stall_since) > per_frame_timeout
        overdue  = time.time() > deadline

        if stalled or overdue:
            reason = "stalled (no new JSON)" if stalled else "total timeout exceeded"
            print(f"  TIMEOUT: {reason} — killing Sapiens")
            proc.kill()
            proc.wait()
            print("STDERR:\n", "".join(stderr_lines)[-3000:])
            return proc.returncode, True

    if proc.returncode != 0:
        print("STDERR:\n", "".join(stderr_lines)[-3000:])
        print("STDOUT:\n", "".join(stdout_lines)[-1000:])
    return proc.returncode, False


for vidf in vfiles:
    name = os.path.splitext(os.path.basename(vidf))[0]

    print(f"working on {name}  ({vfiles.index(vidf)+1}/{len(vfiles)})")

    try:
        vdir    = os.path.join(OUTROOT, name)
        posedir = os.path.join(vdir, "pose_out")
        framedir = os.path.join(vdir, "_frames")
        merged  = os.path.join(vdir, f"{name}_pose.json")
        os.makedirs(posedir, exist_ok=True)
        os.makedirs(framedir, exist_ok=True)

        if os.path.isfile(merged):
            print("  skipping (already merged)")
            continue

        # 1) extract frames (skip already-extracted ones)
        cap = cv2.VideoCapture(vidf)
        fps = cap.get(cv2.CAP_PROP_FPS)
        f = 0
        while True:
            ok, frame = cap.read()
            if not ok: break
            fpath = os.path.join(framedir, f"{f:06d}.png")
            if not os.path.isfile(fpath):  # don't re-extract if resuming
                cv2.imwrite(fpath, frame)
            f += 1
        cap.release()
        total_frames = f
        print(f"  {total_frames} frames @ {fps:.2f} fps")

        # 2) figure out which frames still need processing
        #    a frame is done if its .json exists in posedir
        all_frame_names  = [f"{i:06d}.png" for i in range(total_frames)]
        done_jsons       = {os.path.splitext(os.path.basename(j))[0]
                            for j in glob.glob(os.path.join(posedir, "*.json"))}
        pending          = [fn for fn in all_frame_names
                            if os.path.splitext(fn)[0] not in done_jsons]
        print(f"  {len(done_jsons)} frames already done, {len(pending)} pending")

        MAX_RETRIES = 3
        attempt = 0
        
    

        while pending and attempt < MAX_RETRIES:
            attempt += 1
            print(f"  attempt {attempt}: running Sapiens on {len(pending)} frames")

            # write a text file listing only the pending frames
            pending_list_path = os.path.join(vdir, "_pending.txt")
            with open(pending_list_path, "w") as fh:
                for fn in pending:
                    fh.write(os.path.join(framedir, fn) + "\n")

            command = (f'"{PYTHON}" tools/vis/vis_pose.py '
                       f'"{DETECTOR}" "{CONFIG}" "{CHECKPOINT}" '
                       f'--input "{pending_list_path}" --output "{posedir}" '
                       f'--radius 6 --kpt-thr 0.3 --thickness 4 --bbox-thr 0.9')

            rc, timed_out = run_sapiens(command, cwd=POSE_DIR,
                                        total_frames=len(pending),
                                        per_frame_timeout=600)

            # recompute pending after this attempt
            done_jsons = {os.path.splitext(os.path.basename(j))[0]
                          for j in glob.glob(os.path.join(posedir, "*.json"))}
            pending    = [fn for fn in all_frame_names
                          if os.path.splitext(fn)[0] not in done_jsons]

            if not pending:
                print("  all frames processed")
                break
            elif timed_out:
                print(f"  timed out, {len(pending)} frames still pending — retrying")
            else:
                print(f"  process exited (rc={rc}), {len(pending)} frames still pending — retrying")

        if pending:
            msg = f"{len(pending)} frames never completed after {MAX_RETRIES} attempts"
            print(f"  WARNING: {msg} — skipping to next video")
            sapiens_error.append(f"File {name}: {msg}")
            continue

        # stitch overlay video
        overlays = sorted(glob.glob(os.path.join(posedir, "*.jpg")) +
                          glob.glob(os.path.join(posedir, "*.png")))
        if overlays:
            h, w = cv2.imread(overlays[0]).shape[:2]
            vw = cv2.VideoWriter(os.path.join(vdir, f"{name}_pose.mp4"),
                                 cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
            for p in overlays:
                vw.write(cv2.imread(p))
            vw.release()
        else:
            print("  WARNING: no overlay images found in", posedir)

        # merge per-frame JSONs (exclude the summary _predictions.json if present)
        summary_name = next(
            (os.path.basename(j) for j in glob.glob(os.path.join(posedir, "*.json"))
             if "_predictions" in os.path.basename(j)), None)
        jfiles = sorted(j for j in glob.glob(os.path.join(posedir, "*.json"))
                        if os.path.basename(j) != summary_name)
        combined = {"video": name, "fps": fps, "n_frames": len(jfiles), "frames": {}}
        for jf in jfiles:
            idx = os.path.splitext(os.path.basename(jf))[0]
            with open(jf) as fh:
                combined["frames"][idx] = json.load(fh)
        with open(merged, "w") as fh:
            json.dump(combined, fh)
        print(f"  merged {len(jfiles)} json -> {name}_pose.json")

        # clean up scratch
        shutil.rmtree(posedir)
        shutil.rmtree(framedir)
        os.remove(pending_list_path)
        print("  cleaned scratch")

    except Exception as e:
        print("  ERROR:", e)
        sapiens_error.append(f"File {name} has error: {e}")
        continue

working on GP01_ARBEITEN  (1/78)
  237 frames @ 50.00 fps
  0 frames already done, 237 pending
  attempt 1: running Sapiens on 237 frames
STDERR:
 Traceback (most recent call last):
  File "g:\GtST_FLESH_KinTrajectory\sapiens\pose\tools\vis\vis_pose.py", line 271, in <module>
    main()
  File "g:\GtST_FLESH_KinTrajectory\sapiens\pose\tools\vis\vis_pose.py", line 156, in main
    model = init_model(args.config, args.checkpoint, device=args.device)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "E:\sapiens2\sapiens\pose\src\models\init_model.py", line 36, in init_model
    state_dict = load_file(checkpoint, device="cpu")
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\kadava\AppData\Local\anaconda3\envs\sapiens2\Lib\site-packages\safetensors\torch.py", line 359, in load_file
    with safe_open(filename, framework="pt", device=device, backend=backend) as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


KeyboardInterrupt: 